In [1]:
pip install pandas numpy matplotlib scikit-learn openpyxl xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:

"""
DATASET 3: ROAD NOISE RISK LEVEL PREDICTION
Input: Road_noise_measurements_filled.xlsx
Sheet: Road_noise_measurements

Models:
    LR  - Logistic Regression
    LMR - Linear Regression baseline
    XGB - XGBoost
    RF  - Random Forest

Train/test splits: 80/20, 70/30, 60/40
Robustness levels: 20%, 30%, 40%

Outputs:
    dataset3_output/
        CSV result tables
        figures/
        dataset3_analysis_report.pdf

NOTE:
This follows the documented methodology in the reference PDF.
Exact numerical reproduction is not guaranteed because the original
robustness random seed and every implementation detail are not specified.
"""

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

try:
    from xgboost import XGBClassifier
except ImportError:
    raise SystemExit(
        "Install the required libraries using:\n"
        "pip install pandas numpy matplotlib scikit-learn openpyxl xgboost"
    )


# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_FILE = "Road_noise_measurements_filled (2).xlsx"
SHEET_NAME = "Road_noise_measurements"

OUTPUT_DIR = Path("dataset3_output")
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

FEATURES = [
    "AVG_NOISE_LVL",
    "MEU",
    "Duration_hours",
    "Cumulative_MEU"
]

TARGET = "Risk_Level"

CLASS_NAMES = ["Low", "Moderate", "High"]

CLASS_TO_CODE = {
    "Low": 0,
    "Moderate": 1,
    "High": 2
}

SPLITS = [
    ("80/20", 0.20),
    ("70/30", 0.30),
    ("60/40", 0.40)
]

PERTURBATIONS = [0.20, 0.30, 0.40]

RANDOM_STATE = 42
PERTURBATION_SEED = 42


# ============================================================
# 2. UTILITY FUNCTIONS
# ============================================================

def save_csv(dataframe, filename):
    """Save a DataFrame in the output directory."""
    dataframe.to_csv(
        OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )


def calculate_metrics(y_true, y_pred):
    """Return classification metrics as percentages."""
    return {
        "Accuracy": 100 * accuracy_score(y_true, y_pred),

        "Precision": 100 * precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "Recall": 100 * recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "F1": 100 * f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }


def save_figure(fig, filename):
    """Save a figure as a high-resolution PNG."""
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / filename,
        dpi=180,
        bbox_inches="tight"
    )
    plt.close(fig)


def create_figure(title, xlabel=None, ylabel=None):
    fig, ax = plt.subplots(figsize=(9, 5.5))

    ax.set_title(title, fontsize=13)

    if xlabel:
        ax.set_xlabel(xlabel)

    if ylabel:
        ax.set_ylabel(ylabel)

    ax.grid(True, alpha=0.25)

    return fig, ax


def add_table_to_pdf(pdf, title, dataframe):
    """Add a formatted DataFrame as a PDF page."""
    fig, ax = plt.subplots(figsize=(11.7, 8.3))

    ax.axis("off")
    ax.set_title(title, fontsize=15, pad=20)

    display_df = dataframe.copy()

    for column in display_df.columns:
        if pd.api.types.is_float_dtype(display_df[column]):
            display_df[column] = display_df[column].map(
                lambda value:
                "" if pd.isna(value) else f"{value:.3f}"
            )

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.4)

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 3. LOAD AND VALIDATE DATASET
# ============================================================

if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_FILE}\n"
        "Keep the Excel file in the same folder as this script."
    )

df_raw = pd.read_excel(
    INPUT_FILE,
    sheet_name=SHEET_NAME
)

print("=" * 65)
print("DATASET 3: ROAD NOISE RISK LEVEL PREDICTION")
print("=" * 65)

print("Original rows:", len(df_raw))
print("Original columns:", len(df_raw.columns))

required_columns = FEATURES + [TARGET]

missing_columns = [
    column for column in required_columns
    if column not in df_raw.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing from dataset: {missing_columns}"
    )

df = df_raw.copy()

# Convert feature columns to numeric.
for column in FEATURES:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    df[column] = df[column].replace(
        [np.inf, -np.inf],
        np.nan
    )

# Normalize class labels.
df[TARGET] = (
    df[TARGET]
    .astype("string")
    .str.strip()
    .str.title()
)

# Keep valid target classes only.
df = df[
    df[TARGET].isin(CLASS_TO_CODE)
].copy()

# Remove completely empty rows and duplicate records.
df = df.dropna(how="all")
df = df.drop_duplicates()
df = df.reset_index(drop=True)

print("Rows used after cleaning:", len(df))
print("\nMissing feature values:")
print(df[FEATURES].isna().sum())


# ============================================================
# 4. DATASET DESCRIPTION AND CLASS DISTRIBUTION
# ============================================================

class_counts = (
    df[TARGET]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

class_distribution = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Number of Samples": [
        int(class_counts[class_name])
        for class_name in CLASS_NAMES
    ]
})

class_distribution["Percentage"] = (
    100 * class_distribution["Number of Samples"] / len(df)
)

save_csv(
    class_distribution,
    "dataset3_class_distribution.csv"
)

print("\nClass distribution:")
print(class_distribution.to_string(index=False))


# Calculate AVG_NOISE_LVL ranges for each class.
range_rows = []

for class_name in CLASS_NAMES:
    values = df.loc[
        df[TARGET] == class_name,
        "AVG_NOISE_LVL"
    ].dropna()

    range_rows.append({
        "Class": class_name,
        "AVG_NOISE_LVL_min_dB": (
            values.min() if len(values) else np.nan
        ),
        "AVG_NOISE_LVL_max_dB": (
            values.max() if len(values) else np.nan
        ),
        "Samples": int((df[TARGET] == class_name).sum())
    })

noise_ranges = pd.DataFrame(range_rows)

save_csv(
    noise_ranges,
    "dataset3_noise_range_by_class.csv"
)

print("\nNoise level ranges by class:")
print(noise_ranges.to_string(index=False))


# Feature quality summary.
feature_quality = pd.DataFrame({
    "Feature": FEATURES,

    "Missing after numeric conversion": [
        int(df[column].isna().sum())
        for column in FEATURES
    ],

    "Unique non-missing values": [
        int(df[column].nunique(dropna=True))
        for column in FEATURES
    ],

    "Constant feature": [
        df[column].nunique(dropna=True) <= 1
        for column in FEATURES
    ]
})

save_csv(
    feature_quality,
    "dataset3_feature_quality.csv"
)

print("\nFeature quality:")
print(feature_quality.to_string(index=False))


# ============================================================
# 5. MODEL DEFINITIONS
# ============================================================

def create_models():
    return {
        "LR": LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        ),

        "LMR": LinearRegression(),

        "XGB": XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.1,
            objective="multi:softprob",
            num_class=3,
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        ),

        "RF": RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }


def predict_class(model_name, model, features):
    predictions = model.predict(features)

    # Linear Regression produces continuous values.
    # Round and clip them to valid class codes.
    if model_name == "LMR":
        predictions = np.rint(predictions).astype(int)
        predictions = np.clip(predictions, 0, 2)

    return np.asarray(predictions, dtype=int)


# ============================================================
# 6. PREPROCESSING
# ============================================================

def fit_preprocessing(X_train, X_test):
    """
    Fit preprocessing using training data only:
      1. Median imputation
      2. IQR winsorization using 3 * IQR
      3. Standard scaling
    """

    imputer = SimpleImputer(strategy="median")

    train_imputed = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=FEATURES,
        index=X_train.index
    )

    test_imputed = pd.DataFrame(
        imputer.transform(X_test),
        columns=FEATURES,
        index=X_test.index
    )

    bounds = {}
    bound_rows = []

    train_clipped = train_imputed.copy()
    test_clipped = test_imputed.copy()

    for column in FEATURES:
        q1 = train_imputed[column].quantile(0.25)
        q3 = train_imputed[column].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 3 * iqr
        upper = q3 + 3 * iqr

        # Preserve constant features without clipping.
        if not np.isfinite(iqr) or iqr == 0:
            lower = -np.inf
            upper = np.inf

        bounds[column] = (lower, upper)

        train_count = int(
            (
                (train_imputed[column] < lower)
                | (train_imputed[column] > upper)
            ).sum()
        )

        test_count = int(
            (
                (test_imputed[column] < lower)
                | (test_imputed[column] > upper)
            ).sum()
        )

        train_clipped[column] = (
            train_imputed[column].clip(lower, upper)
        )

        test_clipped[column] = (
            test_imputed[column].clip(lower, upper)
        )

        bound_rows.append({
            "Feature": column,
            "Lower bound": (
                lower if np.isfinite(lower) else np.nan
            ),
            "Upper bound": (
                upper if np.isfinite(upper) else np.nan
            ),
            "Clipped (train)": train_count,
            "Clipped (test)": test_count
        })

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(train_clipped)
    test_scaled = scaler.transform(test_clipped)

    return (
        train_scaled,
        test_scaled,
        imputer,
        bounds,
        scaler,
        pd.DataFrame(bound_rows)
    )


# ============================================================
# 7. TRAIN AND EVALUATE MODELS
# ============================================================

X_all = df[FEATURES].copy()

y_all = (
    df[TARGET]
    .map(CLASS_TO_CODE)
    .astype(int)
)

all_metrics = []
all_robustness = []
all_confusion_matrices = []
all_classification_reports = []
all_bounds = []

for split_name, test_size in SPLITS:

    print("\n" + "=" * 65)
    print("TRAIN/TEST SPLIT:", split_name)
    print("=" * 65)

    X_train, X_test, y_train, y_test = train_test_split(
        X_all,
        y_all,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y_all
    )

    (
        X_train_processed,
        X_test_processed,
        imputer,
        bounds,
        scaler,
        bounds_table
    ) = fit_preprocessing(X_train, X_test)

    bounds_table.insert(0, "Split", split_name)
    all_bounds.append(bounds_table)

    models = create_models()
    fitted_models = {}

    for model_name, model in models.items():

        print(f"\nTraining {model_name}...")

        model.fit(
            X_train_processed,
            y_train
        )

        fitted_models[model_name] = model

        predictions = predict_class(
            model_name,
            model,
            X_test_processed
        )

        metrics = calculate_metrics(
            y_test,
            predictions
        )

        all_metrics.append({
            "Split": split_name,
            "Model": model_name,
            **metrics
        })

        print(
            f"{model_name}: "
            f"Accuracy={metrics['Accuracy']:.2f}%, "
            f"Precision={metrics['Precision']:.2f}%, "
            f"Recall={metrics['Recall']:.2f}%, "
            f"F1={metrics['F1']:.2f}%"
        )

        # Store confusion matrix data.
        matrix = confusion_matrix(
            y_test,
            predictions,
            labels=[0, 1, 2]
        )

        for i, actual_class in enumerate(CLASS_NAMES):
            for j, predicted_class in enumerate(CLASS_NAMES):
                all_confusion_matrices.append({
                    "Split": split_name,
                    "Model": model_name,
                    "Actual": actual_class,
                    "Predicted": predicted_class,
                    "Count": int(matrix[i, j])
                })

        # Store detailed classification report.
        report = classification_report(
            y_test,
            predictions,
            labels=[0, 1, 2],
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )

        for label, values in report.items():
            if isinstance(values, dict):
                all_classification_reports.append({
                    "Split": split_name,
                    "Model": model_name,
                    "Class/average": label,
                    **values
                })

        # Generate confusion matrix figures for 80/20.
        if split_name == "80/20":

            fig, ax = plt.subplots(figsize=(6.5, 5.5))

            image = ax.imshow(matrix, interpolation="nearest")
            fig.colorbar(image, ax=ax)

            ax.set_title(
                f"Dataset 3 - Confusion Matrix: {model_name} (80/20)"
            )

            ax.set_xticks(range(3), CLASS_NAMES)
            ax.set_yticks(range(3), CLASS_NAMES)

            ax.set_xlabel("Predicted class")
            ax.set_ylabel("Actual class")

            threshold = matrix.max() / 2

            for i in range(3):
                for j in range(3):
                    ax.text(
                        j,
                        i,
                        str(matrix[i, j]),
                        ha="center",
                        va="center",
                        color=(
                            "white"
                            if matrix[i, j] > threshold
                            else "black"
                        )
                    )

            save_figure(
                fig,
                f"confusion_matrix_{model_name}_80_20.png"
            )


    # ========================================================
    # 8. ROBUSTNESS TESTING
    # ========================================================

    # Use the training set's standard deviation to scale noise.
    training_std = (
        X_train.std(ddof=0)
        .fillna(0)
        .to_numpy()
    )

    # Impute missing values in the raw test features first.
    test_imputed = pd.DataFrame(
        imputer.transform(X_test),
        columns=FEATURES,
        index=X_test.index
    )

    for perturbation in PERTURBATIONS:

        print(
            f"\nRobustness test: {split_name}, "
            f"{int(perturbation * 100)}% noise"
        )

        # Fixed seed for reproducibility.
        noise_rng = np.random.default_rng(
            PERTURBATION_SEED
            + int(test_size * 1000)
            + int(perturbation * 100)
        )

        gaussian_noise = (
            noise_rng.normal(size=test_imputed.shape)
            * training_std
            * perturbation
        )

        noisy_test = (
            test_imputed.to_numpy()
            + gaussian_noise
        )

        # Apply the already-fitted training preprocessing.
        noisy_clipped = noisy_test.copy()

        for index, column in enumerate(FEATURES):
            lower, upper = bounds[column]

            noisy_clipped[:, index] = np.clip(
                noisy_clipped[:, index],
                lower,
                upper
            )

        noisy_test_processed = scaler.transform(
            noisy_clipped
        )

        # Do not retrain models on noisy data.
        for model_name, model in fitted_models.items():

            noisy_predictions = predict_class(
                model_name,
                model,
                noisy_test_processed
            )

            noisy_metrics = calculate_metrics(
                y_test,
                noisy_predictions
            )

            all_robustness.append({
                "Split": split_name,
                "Model": model_name,
                "Perturbation": f"{int(perturbation * 100)}%",
                "Perturbation_pct": int(perturbation * 100),
                **noisy_metrics
            })


# ============================================================
# 9. SAVE RESULT TABLES
# ============================================================

metrics_df = pd.DataFrame(all_metrics)
robustness_df = pd.DataFrame(all_robustness)

bounds_df = pd.concat(
    all_bounds,
    ignore_index=True
)

confusion_df = pd.DataFrame(all_confusion_matrices)

classification_reports_df = pd.DataFrame(
    all_classification_reports
)

save_csv(
    metrics_df,
    "dataset3_metrics_all_splits.csv"
)

save_csv(
    robustness_df,
    "dataset3_robustness_all_splits.csv"
)

save_csv(
    bounds_df,
    "dataset3_outlier_bounds.csv"
)

save_csv(
    confusion_df,
    "dataset3_confusion_matrices.csv"
)

save_csv(
    classification_reports_df,
    "dataset3_classification_reports.csv"
)


# ============================================================
# 10. MODEL CONSISTENCY ANALYSIS
# ============================================================

consistency_rows = []

for model_name in ["LR", "LMR", "XGB", "RF"]:

    model_data = metrics_df[
        metrics_df["Model"] == model_name
    ]

    row = {"Model": model_name}

    for metric in [
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]:
        row[f"Mean {metric}"] = model_data[metric].mean()

        row[f"Std {metric}"] = model_data[metric].std(
            ddof=1
        )

    row["Consistency Score"] = (
        row["Mean F1"] - row["Std F1"]
    )

    consistency_rows.append(row)

consistency_df = pd.DataFrame(
    consistency_rows
).sort_values(
    "Consistency Score",
    ascending=False
)

save_csv(
    consistency_df,
    "dataset3_model_consistency.csv"
)


# ============================================================
# 11. BEST MODEL PER SPLIT
# ============================================================

best_rows = []

for split_name, _ in SPLITS:

    split_data = metrics_df[
        metrics_df["Split"] == split_name
    ].copy()

    highest_f1 = split_data["F1"].max()

    winners = split_data[
        np.isclose(
            split_data["F1"],
            highest_f1,
            atol=1e-10
        )
    ]

    highest_accuracy = winners["Accuracy"].max()

    winners = winners[
        np.isclose(
            winners["Accuracy"],
            highest_accuracy,
            atol=1e-10
        )
    ]

    best_rows.append({
        "Criterion": f"Best model - {split_name}",
        "Model": ", ".join(winners["Model"].tolist()),
        "Weighted F1 (%)": highest_f1,
        "Accuracy (%)": highest_accuracy
    })

best_model_df = pd.DataFrame(best_rows)

save_csv(
    best_model_df,
    "dataset3_best_model_by_split.csv"
)


# ============================================================
# 12. ROBUSTNESS SUMMARY
# ============================================================

robustness_summary_rows = []

for model_name in ["LR", "LMR", "XGB", "RF"]:

    clean_f1 = metrics_df.loc[
        metrics_df["Model"] == model_name,
        "F1"
    ].mean()

    row = {
        "Model": model_name,
        "Mean F1 @0% (clean)": clean_f1
    }

    for level in [20, 30, 40]:

        values = robustness_df.loc[
            (robustness_df["Model"] == model_name)
            & (
                robustness_df["Perturbation_pct"]
                == level
            ),
            "F1"
        ]

        row[f"Mean F1 @{level}%"] = values.mean()

    row["F1 drop 0%->40%"] = (
        row["Mean F1 @0% (clean)"]
        - row["Mean F1 @40%"]
    )

    robustness_summary_rows.append(row)

robustness_summary_df = pd.DataFrame(
    robustness_summary_rows
)

save_csv(
    robustness_summary_df,
    "dataset3_robustness_summary.csv"
)


# ============================================================
# 13. ACCURACY AND F1 GRAPHS
# ============================================================

MODEL_ORDER = ["LR", "LMR", "XGB", "RF"]

for split_name, _ in SPLITS:

    split_data = (
        metrics_df[metrics_df["Split"] == split_name]
        .set_index("Model")
        .reindex(MODEL_ORDER)
    )

    for metric in ["Accuracy", "F1"]:

        fig, ax = create_figure(
            f"Dataset 3 - {metric} (%) - {split_name} split",
            "Model",
            f"{metric} (%)"
        )

        ax.bar(
            split_data.index,
            split_data[metric]
        )

        ax.set_ylim(
            max(0, min(95, split_data[metric].min() - 3)),
            101
        )

        for index, value in enumerate(split_data[metric]):
            ax.text(
                index,
                value + 0.15,
                f"{value:.2f}",
                ha="center",
                fontsize=8
            )

        save_figure(
            fig,
            f"{metric.lower()}_{split_name.replace('/', '_')}.png"
        )

    # Precision versus recall.
    fig, ax = create_figure(
        f"Dataset 3 - Precision vs Recall - {split_name}",
        "Weighted Recall (%)",
        "Weighted Precision (%)"
    )

    for model_name, row in split_data.iterrows():

        ax.scatter(
            row["Recall"],
            row["Precision"],
            s=65,
            label=model_name
        )

        ax.annotate(
            model_name,
            (row["Recall"], row["Precision"]),
            xytext=(5, 5),
            textcoords="offset points"
        )

    ax.legend()

    save_figure(
        fig,
        f"precision_vs_recall_{split_name.replace('/', '_')}.png"
    )


# ============================================================
# 14. ROBUSTNESS GRAPHS FOR EACH SPLIT
# ============================================================

for split_name, _ in SPLITS:

    split_robustness = robustness_df[
        robustness_df["Split"] == split_name
    ]

    for metric in ["Accuracy", "F1"]:

        fig, ax = create_figure(
            f"Dataset 3 - {metric} vs Perturbation - {split_name}",
            "Perturbation level",
            f"{metric} (%)"
        )

        for model_name in MODEL_ORDER:

            clean_value = metrics_df.loc[
                (metrics_df["Split"] == split_name)
                & (metrics_df["Model"] == model_name),
                metric
            ].iloc[0]

            model_data = split_robustness[
                split_robustness["Model"] == model_name
            ].sort_values("Perturbation_pct")

            x_values = (
                [0]
                + model_data["Perturbation_pct"].tolist()
            )

            y_values = (
                [clean_value]
                + model_data[metric].tolist()
            )

            ax.plot(
                x_values,
                y_values,
                marker="o",
                label=model_name
            )

        ax.set_xticks(
            [0, 20, 30, 40],
            ["0% clean", "20%", "30%", "40%"]
        )

        ax.legend()

        save_figure(
            fig,
            f"robustness_{metric.lower()}_"
            f"{split_name.replace('/', '_')}.png"
        )


# ============================================================
# 15. ROBUSTNESS GRAPHS ACROSS ALL SPLITS
# ============================================================

LINE_STYLES = {
    "80/20": "-",
    "70/30": "--",
    "60/40": ":"
}

for metric in ["Accuracy", "F1"]:

    fig, ax = create_figure(
        f"Dataset 3 - {metric} vs Perturbation - All Splits",
        "Perturbation level",
        f"{metric} (%)"
    )

    for split_name, _ in SPLITS:

        for model_name in MODEL_ORDER:

            clean_value = metrics_df.loc[
                (metrics_df["Split"] == split_name)
                & (metrics_df["Model"] == model_name),
                metric
            ].iloc[0]

            model_data = robustness_df[
                (robustness_df["Split"] == split_name)
                & (robustness_df["Model"] == model_name)
            ].sort_values("Perturbation_pct")

            x_values = (
                [0]
                + model_data["Perturbation_pct"].tolist()
            )

            y_values = (
                [clean_value]
                + model_data[metric].tolist()
            )

            ax.plot(
                x_values,
                y_values,
                marker="o",
                linestyle=LINE_STYLES[split_name],
                label=f"{model_name} {split_name}"
            )

    ax.set_xticks(
        [0, 20, 30, 40],
        ["0% clean", "20%", "30%", "40%"]
    )

    ax.legend(fontsize=7, ncol=2)

    save_figure(
        fig,
        f"robustness_{metric.lower()}_all_splits.png"
    )


# ============================================================
# 16. MEAN PERFORMANCE GRAPHS
# ============================================================

for metric in ["Accuracy", "F1"]:

    fig, ax = create_figure(
        f"Dataset 3 - Mean {metric} (%) by Model",
        "Model",
        f"Mean {metric} (%)"
    )

    ordered_values = [
        consistency_df.loc[
            consistency_df["Model"] == model_name,
            f"Mean {metric}"
        ].iloc[0]
        for model_name in MODEL_ORDER
    ]

    ax.bar(
        MODEL_ORDER,
        ordered_values
    )

    ax.set_ylim(
        max(0, min(95, min(ordered_values) - 3)),
        101
    )

    save_figure(
        fig,
        f"mean_{metric.lower()}_by_model.png"
    )


# ============================================================
# 17. PERFORMANCE ACROSS TRAIN/TEST SPLITS
# ============================================================

for metric in ["Accuracy", "F1"]:

    fig, ax = create_figure(
        f"Dataset 3 - {metric} Across Splits",
        "Train/test split",
        f"{metric} (%)"
    )

    for model_name in MODEL_ORDER:

        model_data = (
            metrics_df[metrics_df["Model"] == model_name]
            .set_index("Split")
            .reindex([split[0] for split in SPLITS])
        )

        ax.plot(
            model_data.index,
            model_data[metric],
            marker="o",
            label=model_name
        )

    ax.legend()

    save_figure(
        fig,
        f"{metric.lower()}_across_splits.png"
    )


# ============================================================
# 18. CREATE CONSOLIDATED PDF REPORT
# ============================================================

report_path = OUTPUT_DIR / "dataset3_analysis_report.pdf"

with PdfPages(report_path) as pdf:

    # Cover page.
    fig, ax = plt.subplots(figsize=(11.7, 8.3))
    ax.axis("off")

    ax.text(
        0.5, 0.84,
        "ROAD NOISE RISK LEVEL PREDICTION",
        ha="center",
        fontsize=20,
        weight="bold"
    )

    ax.text(
        0.5, 0.76,
        "Dataset 3 - Complete Comparative Analysis",
        ha="center",
        fontsize=15
    )

    ax.text(
        0.5, 0.68,
        f"Input: {INPUT_FILE}\nSheet: {SHEET_NAME}",
        ha="center",
        fontsize=11
    )

    details = (
        f"Rows used: {len(df)}\n"
        f"Features: {', '.join(FEATURES)}\n"
        f"Target: {TARGET}\n"
        "Models: LR, LMR, XGB, RF\n"
        "Train/test splits: 80/20, 70/30, 60/40\n"
        "Robustness: 20%, 30%, 40% Gaussian perturbations\n"
        f"Random state: {RANDOM_STATE}"
    )

    ax.text(
        0.08, 0.52,
        details,
        fontsize=12,
        va="top",
        linespacing=1.7
    )

    ax.text(
        0.08, 0.18,
        "Interpretation caution: the reference report states that "
        "AVG_NOISE_LVL ranges do not overlap between risk classes. "
        "Near-perfect performance may therefore reflect thresholds "
        "already encoded in the target labels and should not automatically "
        "be interpreted as generalization to unseen locations.",
        fontsize=10,
        wrap=True,
        va="top"
    )

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # Add tables.
    add_table_to_pdf(
        pdf,
        "Dataset 3 - Class Distribution",
        class_distribution
    )

    add_table_to_pdf(
        pdf,
        "Dataset 3 - Noise Level Ranges",
        noise_ranges
    )

    add_table_to_pdf(
        pdf,
        "Dataset 3 - Feature Quality",
        feature_quality
    )

    add_table_to_pdf(
        pdf,
        "Dataset 3 - Outlier Bounds",
        bounds_df
    )

    add_table_to_pdf(
        pdf,
        "Dataset 3 - Metrics Across Splits",
        metrics_df
    )

    add_table_to_pdf(
        pdf,
        "Dataset 3 - Robustness Results",
        robustness_df
    )

    add_table_to_pdf(
        pdf,
        "Dataset 3 - Robustness Summary",
        robustness_summary_df
    )

    add_table_to_pdf(
        pdf,
        "Dataset 3 - Model Consistency",
        consistency_df
    )

    add_table_to_pdf(
        pdf,
        "Dataset 3 - Best Model by Split",
        best_model_df
    )

    # Append all generated figures to the report.
    for image_path in sorted(FIGURE_DIR.glob("*.png")):

        image = plt.imread(image_path)

        fig, ax = plt.subplots(figsize=(11, 7.2))

        ax.imshow(image)
        ax.axis("off")

        ax.set_title(
            image_path.stem.replace("_", " "),
            fontsize=12
        )

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)


# ============================================================
# 19. FINAL CONSOLE OUTPUT
# ============================================================

print("\n" + "=" * 65)
print("DATASET 3 ANALYSIS COMPLETED")
print("=" * 65)

print("\nOutput folder:")
print(OUTPUT_DIR.resolve())

print("\nPDF report:")
print(report_path.resolve())

print("\nMODEL PERFORMANCE ACROSS ALL SPLITS")
print(
    metrics_df.round(3).to_string(index=False)
)

print("\nROBUSTNESS SUMMARY")
print(
    robustness_summary_df.round(3).to_string(index=False)
)

print("\nBEST MODEL BY SPLIT")
print(
    best_model_df.round(3).to_string(index=False)
)

print("\nMODEL CONSISTENCY")
print(
    consistency_df.round(3).to_string(index=False)
)

print("\nAll result tables and graphs have been saved.")


DATASET 3: ROAD NOISE RISK LEVEL PREDICTION
Original rows: 7386
Original columns: 14
Rows used after cleaning: 7386

Missing feature values:
AVG_NOISE_LVL     0
MEU               0
Duration_hours    0
Cumulative_MEU    0
dtype: int64

Class distribution:
   Class  Number of Samples  Percentage
     Low                988   13.376659
Moderate               3488   47.224479
    High               2910   39.398863

Noise level ranges by class:
   Class  AVG_NOISE_LVL_min_dB  AVG_NOISE_LVL_max_dB  Samples
     Low                  38.4                  54.9      988
Moderate                  55.0                  65.0     3488
    High                  65.1                  77.7     2910

Feature quality:
       Feature  Missing after numeric conversion  Unique non-missing values  Constant feature
 AVG_NOISE_LVL                                 0                        320             False
           MEU                                 0                          1              True
Duratio